## Organized reproduction copy

Run the setup cell first, then Run All. Original analysis cells and saved UCLA outputs below are unchanged. The setup only resolves the organized folder paths. Inputs are linked from `04_results/reproduction_inputs`; new runs write to `04_results/reruns`, preserving the supplied results. Notebook 01 reconstructs the feature table; notebooks 02–03 use the included validated table. Older mapping-unverified wording is historical: all 2,661 mappings were subsequently verified in notebook 03.


In [ ]:
# Folder setup only; the original analysis cells below are unchanged.
from pathlib import Path
import os
_here = Path.cwd().resolve()
_project = next((p for p in [_here, *_here.parents] if (p/'01_original_data').is_dir() and (p/'02_notebooks').is_dir() and (p/'04_results').is_dir()), None)
if _project is None:
    raise FileNotFoundError('Start Jupyter inside the organized wallet_profit_research folder.')
os.chdir(_project/'02_notebooks')
os.environ['WALLET_MODEL_ROOT'] = str(Path.cwd())
os.environ['WALLET_VALIDATION_ROOT'] = str(Path.cwd())
os.environ['WALLET_MODEL_MODE'] = 'production'
os.environ['WALLET_VALIDATION_MODE'] = 'production'
os.environ['WALLET_MODEL_OUTPUT'] = str(_project/'04_results/reruns/modeling/production')
os.environ['WALLET_VALIDATION_OUTPUT'] = str(_project/'04_results/reruns/validation/production')
assert (Path('data')/'wallet_market_candidates_full_week.parquet').is_file()
assert len(list((Path('data')/'weekly_trades').glob('trades-*.parquet'))) == 7
print('Inputs:', Path('data').resolve())
print('New outputs:', (_project/'04_results/reruns').resolve())


# 全周逐笔交易整合与钱包历史特征

新增七个原始文件使我们能够研究**所提供的一周、所有观察市场**内的钱包活动、市场广度和买入集中度。它不等于终身经验，也不证明全球交易完整性。先重现原有 2,462 个试点组合，再扩展到 2,661 个候选市场的 96,184 个钱包–市场组合，计算条件性毛利；映射、期初成本、手续费和非交易持仓变化仍未核验，真实净利润标签全部未知。

从项目目录运行全部单元；需要 data/weekly_trades/、原有 data/ 试点与市场汇总文件。所有运算均内嵌。DuckDB 单线程、3GB 内存上限，允许磁盘溢写；原始交易不会整体载入 pandas。先重现试点，再审核全周并替换局部试点历史。输出写入 `results/full_week/`。扩展市场标签仍仅为假设下毛利；没有真实净利润标签、模型或 p 值。


以下为原有三个试点的基线定义与重现；本节的样本数和三个市场限制仅针对试点。全周及候选扩展见后续单元。

## 原有试点基线重现（下方另有候选扩展）

本笔记本处理三个试点市场，输出一份 2,462 行的钱包–市场表。预测点是首次观察时间戳的整批成交之后。`conditional_profit_label` 仅在零期初等假设下、且观察库存不为负时给出条件性毛利标签；117 个负库存组合留空。`confirmed_net_profit_label` 全部未知，不能视为真实净利润标签。结果映射、手续费、期初成本与非交易持仓变化均未核验。

在项目目录放置 `data/`，用 Python 3/Jupyter 从头运行全部单元即可重现；代码均内嵌于本笔记本。最终整合文件写入 results/full_week/。三个市场不足以支持可靠的独立显著性结论。

This notebook preserves all 6,378 original rows and produces 2,462 wallet–market rows. Run all cells from the project directory with `data/` present. All analysis code is inline; no other analysis script is imported. Raw identifiers are intentionally retained in the requested export.

The prediction point is **after all fills at the wallet's first observed timestamp in this market**. This is observed-window entry, not proven first-ever entry. Current-wallet features use that batch; market context uses strictly earlier timestamps. Same-timestamp activity by other wallets is excluded. All-market history is unavailable. Explicit pilot-history features below cover only the other two supplied markets and must not be interpreted as complete wallet experience.

Accounting is explicitly conditional: zero opening inventory, complete observed trading, no unrecorded position changes, no fees, and an **unverified** interpretation that `price_at_close` pays outcome index 0 and its complement pays index 1. Reversing that mapping is a sensitivity scenario, not a competing verified label. Neither scenario establishes true net profit. All confirmed net-profit labels remain unknown.


In [ ]:
from pathlib import Path
import sys, os, socket, json, hashlib, platform, time
import numpy as np
import pandas as pd
from IPython.display import display
START = time.perf_counter()
CONFIG = {'data_dir': 'data', 'output_csv': 'wallet_market_pilot.csv',
          'profit_epsilon': 1e-6, 'inventory_epsilon': 1e-6,
          'recent_window_minutes': 5, 'expected_rows': 6378,
          'expected_pairs': 2462, 'expected_duplicate_excess': 103,
          'expected_negative_inventory_pairs': 117,
          'mapping_verified': False, 'opening_inventory_verified': False,
          'fees_available': False}
DATA = Path(CONFIG['data_dir'])
INPUTS = ['pilot_trades_three_markets.parquet', 'pilot_market_metadata.csv']
provenance = {'configuration': CONFIG, 'python': sys.version, 'pandas': pd.__version__,
              'numpy': np.__version__, 'hostname': socket.gethostname(),
              'scheduler_job_id': os.environ.get('JOB_ID'), 'platform': platform.platform(),
              'inputs': {name: {'sha256': hashlib.sha256((DATA/name).read_bytes()).hexdigest(),
                                 'bytes': (DATA/name).stat().st_size} for name in INPUTS}}
import pyarrow
provenance['pyarrow'] = pyarrow.__version__
trades = pd.read_parquet(DATA/INPUTS[0])
metadata = pd.read_csv(DATA/INPUTS[1])
trades['time'] = pd.to_datetime(trades.time, utc=True)
meta = metadata.set_index('market', verify_integrity=True)
assert len(trades) == CONFIG['expected_rows']
assert trades.groupby(['wallet','market']).ngroups == CONFIG['expected_pairs']
assert trades.duplicated().sum() == CONFIG['expected_duplicate_excess']
assert not trades.isna().any().any()
assert set(trades.side) == {'BUY','SELL'}
assert trades.price.between(0,1).all() and trades['size'].gt(0).all()
assert trades.dollars.ge(0).all()
assert np.allclose(trades.dollars, trades.price*trades['size'], atol=1e-8, rtol=1e-10)
assert set(trades.market) == set(meta.index)
assert len(meta) == 3
print('Input assertions passed. Maximum dollar discrepancy:',
      (trades.dollars-trades.price*trades['size']).abs().max())
display(pd.DataFrame(provenance['inputs']).T)


以下为原有三个试点的基线定义与重现；本节的样本数和三个市场限制仅针对试点。全周及候选扩展见后续单元。

## Formulas and field roles

For each outcome o, signed shares are +size for BUY and −size for SELL. Final observed net shares q[o] sum those signed shares from hypothetical zero opening inventory. Cash flow C = sell dollars − buy dollars. Assumed gross PNL = C + q[0]p[0] + q[1]p[1]. Reversal PNL uses (1−p[0],1−p[1]). No cash redemption is added a second time. Conditional profit = 1[assumed gross PNL > epsilon]; non-profit includes numerical ties. Real net profit additionally needs actual fees, complete transfers/operations and justified opening cost basis.

Every `feature_` column is available from observed trades at prediction time; `audit_only_` fields use final/full-window information and must never enter a prediction model. `label_` fields are accounting targets/scenarios, not features. Outcome names and assumed payouts are label-context fields from a later metadata snapshot, not historically available features. Nullable `confirmed_net_profit_label` is the desired target, currently entirely missing.

Pilot history H1/H2/H3 are row count, distinct markets and buy dollars from other supplied markets strictly before prediction; log1p versions are included. H4 is sum of squared buy-dollar shares by historical market, missing if no buys; H6/H7 are hours since first/latest observed pilot-history fill. An empty pilot history is not evidence of no all-market history.

Buy VWAP = buy dollars / buy shares. Buy direction concentration = |2 × outcome0 buy-dollar share − 1|; its HHI is (1+D²)/2. No-buy ratios remain missing. Recent relative price compares outcome-specific entry buy VWAP with strictly prior 5-minute market VWAP, weighted by entry buy dollars; if any purchased outcome lacks recent observations, the complete comparison is missing. This is a transaction-price comparison, not slippage. Final negative inventory aggregates equal timestamps before cumulative sums; row order within a timestamp is unknown.


In [ ]:
payout_rows=[]
for market, m in meta.iterrows():
    names=json.loads(m.outcomes)
    assert len(names)==2 and len(set(names))==2
    assert set(trades.loc[trades.market.eq(market),'outcome']) <= set(names)
    assert m.price_at_close in (0,1)
    for k,name in enumerate(names):
        payout_rows.append({'market':market,'outcome_index':k,'outcome':name,
                           'assumed_base_payout':float(m.price_at_close if k==0 else 1-m.price_at_close),
                           'mapping_status':'unverified'})
payouts=pd.DataFrame(payout_rows)
display(payouts)

def accounting(g, outcomes, p0):
    signed=g.assign(signed=np.where(g.side.eq('BUY'),g['size'],-g['size']))
    q=signed.groupby('outcome').signed.sum().reindex(outcomes,fill_value=0)
    cash=g.loc[g.side.eq('SELL'),'dollars'].sum()-g.loc[g.side.eq('BUY'),'dollars'].sum()
    base=cash+q.iloc[0]*p0+q.iloc[1]*(1-p0)
    reverse=cash+q.iloc[0]*(1-p0)+q.iloc[1]*p0
    balances=signed.groupby(['outcome','time']).signed.sum().reset_index().sort_values('time')
    balances['balance']=balances.groupby('outcome').signed.cumsum()
    return q.to_numpy(), cash, base, reverse, min(0.0,balances.balance.min())

rows=[]
for market,g in trades.groupby('market',sort=True):
    outcomes=json.loads(meta.loc[market,'outcomes']);p0=float(meta.loc[market,'price_at_close'])
    for wallet,w in g.groupby('wallet',sort=True):
        tau=w.time.min();batch=w[w.time.eq(tau)]
        prior=g[g.time.lt(tau)];recent=prior[prior.time.ge(tau-pd.Timedelta(minutes=CONFIG['recent_window_minutes']))]
        assert batch.time.eq(tau).all() and (prior.empty or prior.time.max()<tau)
        buy=batch[batch.side.eq('BUY')];sell=batch[batch.side.eq('SELL')]
        B=buy.dollars.sum();S=sell.dollars.sum();Q=buy['size'].sum()
        r={'wallet':wallet,'market':market,'prediction_time':tau,
           'feature_entry_buy_dollars':B,'feature_entry_log_buy_dollars':np.log1p(B),
           'feature_entry_sell_dollars':S,'feature_entry_total_dollars':B+S,
           'feature_entry_sell_dollar_fraction':S/(B+S) if B+S else np.nan,
           'feature_entry_buy_shares':Q,'feature_entry_buy_vwap':B/Q if Q else np.nan,
           'feature_entry_fill_rows':len(batch),'feature_entry_log_fill_rows':np.log1p(len(batch)),
           'feature_entry_has_sell':int(len(sell)>0),'feature_entry_positive_sell_dollars':int(S>0),
           'feature_entry_no_buy':int(Q==0),'feature_entry_minutes_since_last_trade':0.0,
           'feature_strict_prior_market_dollars':prior.dollars.sum(),
           'feature_strict_prior_market_log_dollars':np.log1p(prior.dollars.sum()),
           'feature_strict_prior_market_wallets':prior.wallet.nunique(),
           'feature_strict_prior_market_log_wallets':np.log1p(prior.wallet.nunique()),
           'feature_observed_market_age_hours':(tau-g.time.min()).total_seconds()/3600,
           'feature_hours_since_observation_window_start':(tau-pd.Timestamp('2026-09-29',tz='UTC')).total_seconds()/3600,
           'feature_entry_utc_hour':tau.hour,'feature_entry_utc_weekday':tau.dayofweek,
           'feature_entry_utc_date':str(tau.date()),
           'audit_only_strict_prior_latest_time':prior.time.max(),
           'audit_only_first_batch_latest_time':batch.time.max()}
        hist=trades[trades.wallet.eq(wallet)&trades.market.ne(market)&trades.time.lt(tau)]
        hb=hist[hist.side.eq('BUY')];ha=hb.groupby('market').dollars.sum();ht=ha.sum()
        r.update({'feature_pilot_history_fill_rows':len(hist),
                  'feature_pilot_history_log_fill_rows':np.log1p(len(hist)),
                  'feature_pilot_history_distinct_timestamps':hist.time.nunique(),
                  'feature_pilot_history_markets':hist.market.nunique(),
                  'feature_pilot_history_log_markets':np.log1p(hist.market.nunique()),
                  'feature_pilot_history_buy_dollars':ht,
                  'feature_pilot_history_log_buy_dollars':np.log1p(ht),
                  'feature_pilot_history_buy_market_hhi':((ha/ht)**2).sum() if ht else np.nan,
                  'feature_pilot_history_observed_span_hours':(tau-hist.time.min()).total_seconds()/3600 if len(hist) else np.nan,
                  'feature_pilot_history_recency_hours':(tau-hist.time.max()).total_seconds()/3600 if len(hist) else np.nan,
                  'feature_pilot_history_no_observed_rows':int(hist.empty),
                  'feature_pilot_history_no_buy':int(ht==0)})
        assert hist.empty or (hist.time.lt(tau).all() and hist.market.ne(market).all())
        relative=0.;has_recent=True
        for k,o in enumerate(outcomes):
            ob=buy[buy.outcome.eq(o)];osell=sell[sell.outcome.eq(o)]
            od=ob.dollars.sum();oq=ob['size'].sum();z=recent[recent.outcome.eq(o)]
            zq=z['size'].sum();zv=z.dollars.sum()/zq if zq else np.nan
            r.update({f'feature_entry_outcome{k}_buy_dollars':od,
                      f'feature_entry_outcome{k}_sell_dollars':osell.dollars.sum(),
                      f'feature_entry_outcome{k}_buy_shares':oq,
                      f'feature_entry_outcome{k}_buy_vwap':od/oq if oq else np.nan,
                      f'feature_entry_outcome{k}_observed_net_shares_zero_start':oq-osell['size'].sum(),
                      f'feature_recent_outcome{k}_vwap':zv,
                      f'feature_recent_outcome{k}_missing':int(zq==0)})
            if od>0:
                if zq: relative+=od/B*(od/oq-zv)
                else:has_recent=False
        share=r['feature_entry_outcome0_buy_dollars']/B if B else np.nan
        d=abs(2*share-1)
        r.update({'feature_entry_outcome0_buy_dollar_share':share,
                  'feature_entry_direction_concentration':d,
                  'feature_entry_direction_hhi':(1+d*d)/2,
                  'feature_entry_both_outcomes_bought':int(all(r[f'feature_entry_outcome{k}_buy_dollars']>0 for k in (0,1))),
                  'feature_entry_relative_recent_price':relative if B and has_recent else np.nan,
                  'feature_entry_relative_recent_price_missing':int(not (B and has_recent))})
        q,cash,base,reverse,minbal=accounting(w,outcomes,p0)
        _,_,dedup,_,_=accounting(w.drop_duplicates(),outcomes,p0)
        dup=int(w.duplicated().sum());negative=minbal < -CONFIG['inventory_epsilon']
        reasons=['settlement_mapping_unverified','opening_inventory_and_cost_basis_unverified',
                 'trade_coverage_unverified','fees_unavailable','transfers_split_merge_redemption_unavailable',
                 'historical_result_availability_unverified']
        if negative: reasons.append('negative_observed_inventory')
        if dup: reasons.append('duplicate_looking_rows_without_unique_trade_id')
        r.update({'label_context_outcome0':outcomes[0],'label_context_outcome1':outcomes[1],
                  'label_context_assumed_payout_outcome0':p0,'label_context_assumed_payout_outcome1':1-p0,
                  'label_context_mapping_status':'unverified',
                  'label_final_observed_buy_dollars':w.loc[w.side.eq('BUY'),'dollars'].sum(),
                  'label_final_observed_sell_dollars':w.loc[w.side.eq('SELL'),'dollars'].sum(),
                  'label_final_observed_net_shares_outcome0_zero_start':q[0],
                  'label_final_observed_net_shares_outcome1_zero_start':q[1],
                  'label_final_observed_cashflow':cash,
                  'label_assumed_zero_start_gross_pnl':base,
                  'label_diagnostic_assumed_pnl_sign':int(base>CONFIG['profit_epsilon']),
                  'conditional_profit_label':pd.NA if negative else int(base>CONFIG['profit_epsilon']),
                  'label_reversed_mapping_gross_pnl':reverse,
                  'label_reversed_mapping_diagnostic_profit_sign':int(reverse>CONFIG['profit_epsilon']),
                  'label_dedup_assumption_gross_pnl':dedup,
                  'label_dedup_assumption_diagnostic_profit_sign':int(dedup>CONFIG['profit_epsilon']),
                  'confirmed_net_profit_label':pd.NA,'label_status':'unknown_unverified',
                  'label_unknown_reasons':';'.join(reasons),
                  'audit_only_final_fill_rows':len(w),'audit_only_duplicate_excess':dup,
                  'audit_only_min_observed_inventory_zero_start':minbal,
                  'audit_only_negative_observed_inventory':int(negative),
                  'audit_only_final_observed_time':w.time.max()})
        assert np.isclose(base+reverse,2*cash+sum(q))
        rows.append(r)
result=pd.DataFrame(rows)
for scenario, column in [('base','label_assumed_zero_start_gross_pnl'),('reverse','label_reversed_mapping_gross_pnl'),('dedup','label_dedup_assumption_gross_pnl')]:
    result['label_'+scenario+'_gross_state']=np.select([result[column]>CONFIG['profit_epsilon'],result[column]<-CONFIG['profit_epsilon']],['profit','loss'],default='breakeven')
result['conditional_profit_label']=result.conditional_profit_label.astype('Int64')
result['confirmed_net_profit_label']=result.confirmed_net_profit_label.astype('Int64')
for reason in ['settlement_mapping_unverified','opening_inventory_and_cost_basis_unverified','trade_coverage_unverified','fees_unavailable','transfers_split_merge_redemption_unavailable','historical_result_availability_unverified','negative_observed_inventory','duplicate_looking_rows_without_unique_trade_id']:
    result['label_reason_'+reason]=result.label_unknown_reasons.str.split(';').map(lambda items: int(reason in items))
FEATURE_COLUMNS=[c for c in result if c.startswith('feature_')]
print('Computed',len(result),'pairs and',len(FEATURE_COLUMNS),'feature columns')


以下为原有三个试点的基线定义与重现；本节的样本数和三个市场限制仅针对试点。全周及候选扩展见后续单元。

## Proof checks and descriptive summaries

The checks below establish computation boundaries and internal identities; they do not verify source completeness or actual settlement mapping. Missing feature values are reported without filling. Direction HHI and concentration, or sell fraction and net-flow fraction, are alternative encodings, not independent evidence. Market/date controls can be collinear in this tiny pilot.


In [ ]:
assert len(result)==2462 and not result[['wallet','market']].duplicated().any()
assert result.confirmed_net_profit_label.isna().all()
assert result.conditional_profit_label.notna().sum()==2345
assert result.loc[result.audit_only_negative_observed_inventory.eq(1),'conditional_profit_label'].isna().all()
assert result.audit_only_negative_observed_inventory.sum()==117
assert result.audit_only_duplicate_excess.sum()==103
assert result.audit_only_first_batch_latest_time.eq(result.prediction_time).all()
known_prior=result.audit_only_strict_prior_latest_time.notna()
assert (result.loc[known_prior,'audit_only_strict_prior_latest_time']<result.loc[known_prior,'prediction_time']).all()
assert not any(c.startswith(('audit_only_','label_')) for c in FEATURE_COLUMNS)
assert set(result.label_context_mapping_status)=={'unverified'}
assert result.label_unknown_reasons.str.contains('fees_unavailable').all()
assert np.allclose(result.label_final_observed_cashflow,
                   result.label_final_observed_sell_dollars-result.label_final_observed_buy_dollars)
assert np.allclose(result.label_assumed_zero_start_gross_pnl,
    result.label_final_observed_cashflow+
    result.label_final_observed_net_shares_outcome0_zero_start*result.label_context_assumed_payout_outcome0+
    result.label_final_observed_net_shares_outcome1_zero_start*result.label_context_assumed_payout_outcome1)
# Hand calculation: buy 10 A at .4, sell 3 A at .6, buy 2 B at .25.
fixture=pd.DataFrame({'side':['BUY','SELL','BUY'],'outcome':['A','A','B'],
    'size':[10.,3.,2.],'dollars':[4.,1.8,.5],
    'time':pd.to_datetime(['2026-01-01','2026-01-02','2026-01-01'],utc=True)})
fq,fc,fp,fr,fmin=accounting(fixture,['A','B'],1.)
assert np.allclose(fq,[7,2]) and np.isclose(fc,-2.7)
assert np.isclose(fp,4.3) and np.isclose(fr,-.7) and fmin==0
assert np.isclose(result.label_final_observed_buy_dollars.sum(),trades.loc[trades.side.eq('BUY'),'dollars'].sum())
assert np.isclose(result.label_final_observed_sell_dollars.sum(),trades.loc[trades.side.eq('SELL'),'dollars'].sum())
assert result.audit_only_duplicate_excess.sum()==len(trades)-len(trades.drop_duplicates())
print('Duplicate-affected pairs:',result.audit_only_duplicate_excess.gt(0).sum())
# Strong no-lookahead test: features recomputed from prefix-only data for representative entries.
# Each source used by feature construction is explicitly restricted to batch or strict-prior rows.
for _,r in result.groupby('market').head(3).iterrows():
    g=trades[trades.market.eq(r.market)];prefix=g[g.time.le(r.prediction_time)]
    own=prefix[prefix.wallet.eq(r.wallet)];prior=prefix[prefix.time.lt(r.prediction_time)]
    # Adding a hypothetical later fill cannot change either prediction source.
    extra=g.iloc[[0]].copy();extra['time']=r.prediction_time+pd.Timedelta(days=1)
    extra['dollars']=1e12
    augmented=pd.concat([g,extra],ignore_index=True)
    pd.testing.assert_frame_equal(g[g.time.le(r.prediction_time)].reset_index(drop=True),
        augmented[augmented.time.le(r.prediction_time)].reset_index(drop=True), check_dtype=False)
    assert own.time.eq(r.prediction_time).all()
    assert len(own)==r.feature_entry_fill_rows
    assert np.isclose(prior.dollars.sum(),r.feature_strict_prior_market_dollars)
    assert prior.wallet.nunique()==r.feature_strict_prior_market_wallets
summary=result.groupby('market').agg(
    wallet_market_pairs=('wallet','size'), observed_rows=('audit_only_final_fill_rows','sum'),
    duplicate_excess=('audit_only_duplicate_excess','sum'),
    negative_inventory_pairs=('audit_only_negative_observed_inventory','sum'),
    assumed_gross_pnl=('label_assumed_zero_start_gross_pnl','sum'),
    reversed_mapping_gross_pnl=('label_reversed_mapping_gross_pnl','sum'),
    dedup_assumption_gross_pnl=('label_dedup_assumption_gross_pnl','sum'),
    diagnostic_positive_sign_pairs=('label_diagnostic_assumed_pnl_sign','sum'),
    usable_conditional_labels=('conditional_profit_label','count'),
    conditional_profit_pairs=('conditional_profit_label','sum'),
    confirmed_labels=('confirmed_net_profit_label','count'))
summary['mapping_profit_label_changes']=result.assign(changed=result.label_diagnostic_assumed_pnl_sign.ne(result.label_reversed_mapping_diagnostic_profit_sign)).groupby('market').changed.sum()
summary['dedup_profit_label_changes']=result.assign(changed=result.label_diagnostic_assumed_pnl_sign.ne(result.label_dedup_assumption_diagnostic_profit_sign)).groupby('market').changed.sum()
display(summary)
display(result[FEATURE_COLUMNS].describe(include='all').T)
display(result[FEATURE_COLUMNS].isna().sum().rename('missing_rows').to_frame())
preview=['wallet','market','prediction_time','feature_entry_buy_dollars','feature_entry_buy_vwap','feature_entry_direction_concentration','label_assumed_zero_start_gross_pnl','label_base_gross_state','conditional_profit_label','confirmed_net_profit_label','audit_only_negative_observed_inventory','audit_only_duplicate_excess','label_status']
with pd.option_context('display.max_columns',None,'display.width',240):
    display(result[preview].head(12))
print('All proof checks passed; confirmed labels:', result.confirmed_net_profit_label.notna().sum())


以下为原有三个试点的基线定义与重现；本节的样本数和三个市场限制仅针对试点。全周及候选扩展见后续单元。

## Export dictionary

All dollar fields use recorded trade-dollar units; shares use recorded contract shares. Outcome0/1 refer to catalog order and have no common cross-market bullish meaning. Each row below identifies the exact exported field, computational definition, units, and role. The inline construction cell is the executable specification; missing ratios indicate undefined denominators or absent reference trades.


In [ ]:
dictionary=[]
for col in result.columns:
    role='feature' if col.startswith('feature_') else ('retrospective audit; never predictor' if col.startswith('audit_only_') else ('label/scenario/context; never predictor' if col.startswith('label_') or col.endswith('_label') else 'identifier/time'))
    units='unitless'
    if 'dollars' in col or 'cashflow' in col or 'pnl' in col: units='recorded dollars'
    if 'shares' in col or 'inventory' in col: units='contract shares'
    if 'hours' in col: units='hours'
    if 'minutes' in col: units='minutes'
    if 'time' in col and 'timestamps' not in col: units='UTC timestamp'
    if 'vwap' in col or 'relative_recent_price' in col: units='dollars per contract share'
    if any(word in col for word in ['fill_rows','wallets','markets','duplicate_excess','timestamps']): units='count'
    if col.endswith('utc_date'): units='UTC calendar date'
    if 'log_' in col: units='log1p of underlying count or recorded dollars'
    definition=col.replace('feature_','').replace('audit_only_','').replace('label_','').replace('_',' ')
    if 'log_' in col: definition='ln(1 + '+definition.replace('log ','')+')'
    if 'vwap' in col: definition+='; sum dollars / sum shares for indicated subset'
    if 'hhi' in col: definition+='; sum squared buy-dollar shares; missing if no buys'
    if 'no_buy' in col: definition+='; 1 if indicated subset has no buy shares/dollars'
    if 'missing' in col: definition+='; 1 if indicated ratio/reference is undefined'
    if 'pilot_history' in col: definition+='; other supplied markets only, time < prediction time'
    elif col.startswith('feature_recent_'): definition+='; target market, [T-5min,T), same named outcome'
    elif col.startswith('feature_strict_prior_'): definition+='; target market, time < T'
    elif col.startswith('feature_entry_'): definition+='; own first observed timestamp batch only'
    elif col.startswith('label_reason_'): definition+='; 1 if named evidence limitation applies'
    if 'conditional_profit' in col: definition+='; 1 if scenario PNL > configured epsilon; 0 otherwise'
    if col=='conditional_profit_label': definition+='; unknown if negative observed inventory; other accounting assumptions still unverified'
    if col=='confirmed_net_profit_label': definition='Desired actual net-profit target; unknown for all pairs because accounting evidence unavailable'
    if col=='label_assumed_zero_start_gross_pnl': definition='sell dollars - buy dollars + q0*p0 + q1*(1-p0), unverified base payout mapping'
    if col=='label_reversed_mapping_gross_pnl': definition='sell dollars - buy dollars + q0*(1-p0) + q1*p0'
    if col=='label_dedup_assumption_gross_pnl': definition='Base PNL recomputed after full-field drop_duplicates; sensitivity assumption only'
    dictionary.append({'column':col,'role':role,'units':units,'availability':('observed at prediction time; partial pilot scope' if role=='feature' else ('observed identifier/prediction timestamp' if role=='identifier/time' else 'retrospective or scenario; unverified where stated')),'definition':definition})
column_dictionary=pd.DataFrame(dictionary)
with pd.option_context('display.max_rows',None,'display.max_colwidth',180):
    display(column_dictionary)
assert set(column_dictionary.column)==set(result.columns)


以下为原有三个试点的基线定义与重现；本节的样本数和三个市场限制仅针对试点。全周及候选扩展见后续单元。

## Missing data and limits

The conditional profit columns can be used to audit formulas; they cannot serve as verified profitable-wallet outcomes. Positive inventory is not evidence of complete history. Duplicate-looking rows remain in the primary computation; full-field deduplication is only a sensitivity assumption. Future inventory flags must not be used as entry features or silent sample filters. The assumed outcomes shown above are not independently confirmed winners.

The raw `label_diagnostic_assumed_pnl_sign` is arithmetic for all pairs. `conditional_profit_label` is nullable: 117 negative-inventory pairs have no usable conditional label; the other 2,345 are mechanically consistent under the listed assumptions, not confirmed net profit. Each reason is also exported as a binary reason-code column.

Only three markets are present, with 16 wallets appearing in multiple pilot markets. There are too few independent market clusters for reliable general-market significance tests; 2,462 rows do not solve this. No model, p-value, odds ratio, or predictive-performance claim is produced. The next stage requires independently verified accounting and more independent events, with time/event-separated evaluation and training labels available before test prediction times.


In [ ]:
missing_data=pd.DataFrame([
    (1,'Settlement outcome/token mapping','Independent dictionary or source linking settlement field to named outcome/token; current outcome0 mapping unverified','Verified payouts and profit target'),
    (1,'Opening inventory, cost basis and complete ledger','Pre-window positions, earlier fills, transfers, split/merge, redemption records','Final economic profit and inventory reconciliation'),
    (1,'Actual transaction fees and other cash flows','Per-trade fees and supported adjustments; avoid double-counting redeemed value','Confirmed net-profit label'),
    (1,'Unique execution identifiers','Trade/order/chain log identifiers and duplicate policy','Resolve 103 duplicate-looking excess rows'),
    (2,'Full-week all-market wallet trades and earlier history','All seven daily files with consistent schema; longer lookback if available','H0–H4/H6/H7 history features; no pilot-only proxy'),
    (2,'Historical market and result availability','Event window, actual close/result-publication times and historical snapshots','Verify prediction-time result-unknown eligibility; metadata features'),
    (2,'More independent events and event groups','Audited markets, related-event identifiers and labels','Time/event-separated models and sufficient cluster inference'),
    (3,'Versioned topic classification','Historically available categories and reproducible coding','H5 topic concentration'),
    (3,'Historically known settlement/accounting outcomes','Trusted resolved ledger available before each prediction point','H8/H9 historical success/return')
],columns=['priority','missing_data','required_evidence','enables'])
display(missing_data)
# Only the requested row-level export is written as a CSV; tables above stay in notebook outputs.
# Legacy export suppressed; integrated export below.
provenance['elapsed_seconds']=time.perf_counter()-START
provenance['export_rows']=len(result)
provenance['export_columns']=list(result.columns)
provenance['feature_columns']=FEATURE_COLUMNS
# Integrated output hash calculated below.
# Integrated provenance saved below.
print('Preserved pilot accounting recomputed:',len(result))


## 原始全周审核

哈希按文件分块读取；表结构和 footer 数与解码后的实际行数分别核对。文件名日期与实际 UTC 日期独立比较。汇总逐市场比较全部共同字段，整数与时间精确比较、成交额允许预先声明的数值误差。试点使用所有八字段的 multiset（保留重复次数）比较。若失败，停止而不是悄悄修正。


In [ ]:
import duckdb
import pyarrow.parquet as pq
pyarrow.set_cpu_count(1)
pyarrow.set_io_thread_count(1)
WEEK_START=time.perf_counter()
WEEK_CONFIG={'memory_limit':'3GB','threads':1,'volume_atol':1e-5,'volume_rtol':1e-10,
             'expected_decoded_rows':6815814,'scope':'supplied week all observed markets; not lifetime/global coverage'}
OUTPUT_DIR=Path('results/full_week');OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
spill=OUTPUT_DIR/'duckdb_spill_round03';spill.mkdir(exist_ok=True)
con=duckdb.connect()
con.execute("SET threads=1")
con.execute("SET memory_limit='3GB'")
con.execute("SET TimeZone='UTC'")
con.execute("SET temp_directory='results/full_week/duckdb_spill_round03'")
files=sorted((DATA/'weekly_trades').glob('trades-*.parquet'))
assert len(files)==7
manifest=json.loads((DATA/'weekly_trades/download_manifest.json').read_text())
manifest_by_date={r['date']:r for r in manifest}
def sha256_stream(path):
    h=hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda:f.read(1024*1024),b''): h.update(chunk)
    return h.hexdigest()
file_info=[];daily=[];actual_dates=[]
expected_columns=list(trades.columns)
for path in files:
    parquet=pq.ParquetFile(path);name_date=path.stem.replace('trades-','')
    schema={field.name:str(field.type) for field in parquet.schema_arrow}
    assert list(schema)==expected_columns
    info={'file':str(path),'filename_date':name_date,'sha256':sha256_stream(path),
          'bytes':path.stat().st_size,'footer_rows':parquet.metadata.num_rows,'schema':schema}
    con.from_parquet(str(path)).create_view('daily_source',replace=True)
    row=con.execute("""SELECT count(*) AS "rows", min(time) first_time,max(time) last_time,
        count(DISTINCT CAST(time AS DATE)) utc_days,
        count(*) FILTER(WHERE wallet IS NULL OR market IS NULL OR time IS NULL OR side IS NULL OR outcome IS NULL OR price IS NULL OR size IS NULL OR dollars IS NULL) missing_any,
        count(*) FILTER(WHERE side NOT IN ('BUY','SELL')) invalid_side,
        count(*) FILTER(WHERE NOT isfinite(price) OR price<0 OR price>1) invalid_price,
        count(*) FILTER(WHERE NOT isfinite(size) OR size<=0) invalid_size,
        count(*) FILTER(WHERE NOT isfinite(dollars) OR dollars<0) invalid_dollars,
        max(abs(dollars-price*size)) max_dollar_discrepancy,
        count(DISTINCT side) side_domain_count,count(DISTINCT outcome) outcome_domain_count
        FROM daily_source""").df().iloc[0].to_dict()
    row.update({'file':path.name,'filename_date':name_date,'footer_rows':info['footer_rows']})
    days=con.execute('SELECT CAST(time AS DATE) utc_date,count(*) AS "rows" FROM daily_source GROUP BY 1 ORDER BY 1').df()
    days['file']=path.name;actual_dates.append(days)
    row['rows_outside_filename_date']=int(days.loc[days.utc_date.astype(str).ne(name_date),'rows'].sum())
    row['manifest_rows']=manifest_by_date[name_date]['rows']
    row['manifest_status']=manifest_by_date[name_date]['status']
    assert row['rows']==info['footer_rows']==row['manifest_rows']
    assert info['bytes']==manifest_by_date[name_date]['bytes']
    daily.append(row);file_info.append(info)
daily_audit=pd.DataFrame(daily);date_audit=pd.concat(actual_dates,ignore_index=True)
assert daily_audit.rows.sum()==6815814
for c in ['missing_any','invalid_side','invalid_price','invalid_size','invalid_dollars']:
    assert daily_audit[c].sum()==0,c
assert daily_audit.max_dollar_discrepancy.max()<1e-7
con.from_parquet([str(p) for p in files]).create_view('raw_week')
display(daily_audit);display(date_audit)
daily_audit.to_csv('results/full_week/full_week_daily_audit.csv',index=False)
date_audit.to_csv('results/full_week/full_week_file_date_audit.csv',index=False)


In [ ]:
full_summary=con.execute("""SELECT market,count(*) trade_count,sum(dollars) trading_volume,
 min(time) first_observed_time,max(time) last_observed_time,
 count(DISTINCT CAST(time AS DATE)) observed_days,count(DISTINCT wallet) wallet_count
 FROM raw_week GROUP BY market""").df()
old_summary=pd.read_csv(DATA/'weekly_market_summary.csv')
assert not full_summary.market.duplicated().any() and not old_summary.market.duplicated().any()
compare=full_summary.merge(old_summary,on='market',how='outer',suffixes=('_raw','_original'),indicator=True,validate='one_to_one')
assert compare['_merge'].eq('both').all()
for col in ['trade_count','wallet_count','observed_days']:
    compare[col+'_matches']=compare[col+'_raw'].eq(compare[col+'_original'])
for col in ['first_observed_time','last_observed_time']:
    for suffix in ['_raw','_original']: compare[col+suffix]=pd.to_datetime(compare[col+suffix],utc=True)
    compare[col+'_matches']=compare[col+'_raw'].eq(compare[col+'_original'])
compare['trading_volume_difference']=compare.trading_volume_raw-compare.trading_volume_original
compare['trading_volume_matches']=np.isclose(compare.trading_volume_raw,compare.trading_volume_original,
    atol=WEEK_CONFIG['volume_atol'],rtol=WEEK_CONFIG['volume_rtol'])
checks=[c for c in compare if c.endswith('_matches')]
compare['all_fields_match']=compare[checks].all(axis=1)
compare.to_csv('results/full_week/full_week_market_audit.csv',index=False)
assert compare.all_fields_match.all(), 'Original summary mismatch: inspect full_week_market_audit.csv'
assert len(full_summary)==78543
con.register('pilot_market_keys',pd.DataFrame({'market':meta.index}))
raw_pilot=con.execute('SELECT r.* FROM raw_week r INNER JOIN pilot_market_keys p USING(market)').df()
raw_pilot['time']=pd.to_datetime(raw_pilot.time,utc=True)
keys=list(trades.columns)
a=trades.groupby(keys,dropna=False).size().rename('old_multiplicity').reset_index()
b=raw_pilot.groupby(keys,dropna=False).size().rename('raw_multiplicity').reset_index()
multiset=a.merge(b,on=keys,how='outer').fillna({'old_multiplicity':0,'raw_multiplicity':0})
multiset['matches']=multiset.old_multiplicity.eq(multiset.raw_multiplicity)
multiset_report={'old_rows':len(trades),'raw_rows':len(raw_pilot),'distinct_rows':len(multiset),
                 'mismatched_distinct_rows':int((~multiset.matches).sum()),
                 'multiplicity_abs_difference':int((multiset.old_multiplicity-multiset.raw_multiplicity).abs().sum())}
assert multiset.matches.all() and len(raw_pilot)==6378
print('Exact pilot multiset:',multiset_report)
print('Full summary matched all fields for',len(compare),'markets')


## 严格预测前的一周历史

生产路径先只提取试点钱包，再用 DuckDB 连接 2,462 个预测点并按严格时间与非目标市场筛选。填单数、不同时间戳数和市场数包含买卖；买入金额及 HHI 只用 BUY。HHI=Σ(市场买入额/历史总买入额)²，无买入为缺失。span=预测点−首条历史、recency=预测点−最后历史，以小时计；不是钱包年龄。`no_observed_rows` 仅表示这周文件中未观察到更早活动。

独立校验路径逐一使用 pandas 对提取出的试点钱包交易直接过滤，核对全部组合及缺失模式；不重复使用生产 SQL 聚合。旧的 `feature_pilot_history_*` 删除，避免误将局部历史与周历史混用。历史盈利率/收益率暂不可用：其他市场结算映射和历史结果可知时间未核验，禁止拿最终快照冒充预测前已知结果。


In [ ]:
legacy_history=[c for c in result if c.startswith('feature_pilot_history_')]
legacy_coverage=int(result.feature_pilot_history_no_observed_rows.eq(0).sum())
con.register('pilot_wallet_keys',pd.DataFrame({'wallet':result.wallet.unique()}))
con.execute('CREATE TABLE pilot_wallet_week AS SELECT r.* FROM raw_week r INNER JOIN pilot_wallet_keys p USING(wallet)')
cutoffs=result[['wallet','market','prediction_time']].copy().reset_index().rename(columns={'index':'pair_id','market':'target_market','prediction_time':'cutoff'})
con.register('cutoffs',cutoffs)
con.execute("""CREATE TABLE eligible_history AS
 SELECT c.pair_id,r.* FROM pilot_wallet_week r JOIN cutoffs c ON r.wallet=c.wallet
 WHERE r.time<c.cutoff AND r.market<>c.target_market""")
history=con.execute("""WITH general AS (
 SELECT pair_id,count(*) fill_rows,count(DISTINCT time) distinct_timestamps,
 count(DISTINCT market) markets,sum(CASE WHEN side='BUY' THEN dollars ELSE 0 END) buy_dollars,
 min(time) first_time,max(time) latest_time FROM eligible_history GROUP BY pair_id),
 by_market AS(SELECT pair_id,market,sum(dollars) buy FROM eligible_history WHERE side='BUY' GROUP BY pair_id,market),
 square AS(SELECT pair_id,sum(buy*buy) sq FROM by_market GROUP BY pair_id)
 SELECT g.*,s.sq/(g.buy_dollars*g.buy_dollars) buy_market_hhi FROM general g LEFT JOIN square s USING(pair_id)
 ORDER BY pair_id""").df().set_index('pair_id').reindex(result.index)
for c in ['fill_rows','distinct_timestamps','markets','buy_dollars']: history[c]=history[c].fillna(0)
history.loc[history.buy_dollars.eq(0),'buy_market_hhi']=np.nan
for c in ['first_time','latest_time']:history[c]=pd.to_datetime(history[c],utc=True)
history['observed_span_hours']=(result.prediction_time-history.first_time).dt.total_seconds()/3600
history['recency_hours']=(result.prediction_time-history.latest_time).dt.total_seconds()/3600
history['log_recency_hours']=np.log1p(history.recency_hours)
for c in ['fill_rows','distinct_timestamps','markets','buy_dollars']:history['log_'+c]=np.log1p(history[c])
history['no_observed_rows']=history.fill_rows.eq(0).astype(int)
history['no_buy']=history.buy_dollars.eq(0).astype(int)
history['prior_conditional_profit_history_available']=0
# Independent brute-force validation over only the pilot wallets, all 2462 cutoffs.
wallet_week=con.execute('SELECT * FROM pilot_wallet_week').df()
wallet_week['time']=pd.to_datetime(wallet_week.time,utc=True)
by_wallet={w:g for w,g in wallet_week.groupby('wallet',sort=False)}
for i,r in cutoffs.iterrows():
    w=by_wallet[r.wallet];h=w[w.time.lt(r.cutoff)&w.market.ne(r.target_market)]
    hb=h[h.side.eq('BUY')];amount=hb.dollars.sum();market_buys=hb.groupby('market').dollars.sum()
    expected={'fill_rows':len(h),'distinct_timestamps':h.time.nunique(),'markets':h.market.nunique(),
              'buy_dollars':amount,'buy_market_hhi':((market_buys/amount)**2).sum() if amount else np.nan}
    for c,v in expected.items():
        actual=history.loc[i,c]
        assert (pd.isna(v) and pd.isna(actual)) or np.isclose(v,actual,atol=1e-7,rtol=1e-10),(i,c,v,actual)
    for c,v in [('first_time',h.time.min()),('latest_time',h.time.max())]:
        assert (pd.isna(v) and pd.isna(history.loc[i,c])) or v==history.loc[i,c],(i,c)
    assert history.loc[i,'no_observed_rows']==int(h.empty)
    assert history.loc[i,'no_buy']==int(amount==0)
    if len(h):assert h.time.max()<r.cutoff and not h.market.eq(r.target_market).any()
# Boundary fixture: target and same-timestamp records excluded; earlier other-market SELL counts.
fixture=pd.DataFrame({'time':[1,2,3,1],'market':['other','other','other','target'],'side':['SELL','BUY','BUY','BUY']})
assert len(fixture[fixture.time.lt(2)&fixture.market.ne('target')])==1
old_accounting=result[[c for c in result if c.startswith('label_') or c=='confirmed_net_profit_label' or c=='conditional_profit_label']].copy()
result=result.drop(columns=legacy_history).join(history.add_prefix('feature_week_history_'))
pd.testing.assert_frame_equal(old_accounting,result[old_accounting.columns])
assert len(result)==2462 and result.confirmed_net_profit_label.isna().all()
assert result.conditional_profit_label.notna().sum()==2345
assert not any(c.startswith('feature_pilot_history_') for c in result)
assert result.feature_week_history_prior_conditional_profit_history_available.eq(0).all()
print('Independent history validation passed for all',len(result),'pairs')
print('Pilot-only observed-history pairs:',legacy_coverage,'; supplied-week observed-history pairs:',int(history.no_observed_rows.eq(0).sum()))
print('Decoded pilot-wallet weekly rows:',len(wallet_week))


In [ ]:
universe=con.execute("""SELECT count(*) fill_rows,count(DISTINCT market) markets,
 count(DISTINCT wallet) wallets,count(DISTINCT (wallet,market)) wallet_market_pairs FROM raw_week""").df()
full_metadata=pd.read_csv(DATA/'weekly_market_metadata_complete.csv')
assert not full_metadata.market.duplicated().any()
coverage=[]
for name in ['candidate_markets_preliminary.csv','candidate_markets_100plus_wallets.csv']:
    candidate=pd.read_csv(DATA/name)
    con.register('candidate_keys',candidate[['market']])
    counts=con.execute('SELECT count(*) raw_rows,count(DISTINCT (wallet,market)) wallet_market_pairs,count(DISTINCT wallet) wallets FROM raw_week JOIN candidate_keys USING(market)').df().iloc[0].to_dict()
    coverage.append({'candidate_list':name,'listed_markets':len(candidate),'matched_markets':int(candidate.market.isin(full_summary.market).sum()),**counts})
coverage=pd.DataFrame(coverage)
assert coverage.listed_markets.tolist()==[2661,265]
coverage.to_csv('results/full_week/full_week_candidate_coverage.csv',index=False)
print('Full observed universe:');display(universe)
print('Candidate feasibility inventory, not new verified labels:');display(coverage)
print('Markets with metadata rows:',full_summary.market.isin(full_metadata.market).sum())
display(result.groupby('market').agg(pairs=('wallet','size'),
    week_history_pairs=('feature_week_history_no_observed_rows',lambda s:int(s.eq(0).sum())),
    median_history_rows=('feature_week_history_fill_rows','median'),
    median_history_markets=('feature_week_history_markets','median')))


## 字典、输出与下一步

全周数据让历史行为研究更可行，但已结算历史胜率仍不可用。观察周前的持仓/成本、转账、split/merge、费用、redeem、成交 ID、结果/token 映射、真实结果公开时间及独立事件分组仍需补充。2,661/265 只是回顾性候选覆盖，不代表可直接建模。原有三个试点依然只有三个市场簇，不报告稳定显著性。


In [ ]:
dictionary=column_dictionary[~column_dictionary.column.str.startswith('feature_pilot_history_')].copy()
week_definitions={
 'fill_rows':('count','Number of strict-prior supplied-week rows, BUY and SELL'),
 'distinct_timestamps':('count','Distinct UTC timestamps among eligible rows'),
 'markets':('count','Distinct markets among eligible BUY/SELL rows'),
 'buy_dollars':('recorded dollars','Sum dollars where side=BUY'),
 'buy_market_hhi':('unitless','Sum market buy-dollar squares / total buy dollars squared; missing if zero buys'),
 'first_time':('UTC timestamp','Minimum eligible history timestamp; missing if empty'),
 'latest_time':('UTC timestamp','Maximum eligible history timestamp; missing if empty'),
 'observed_span_hours':('hours','(Prediction time - first eligible history time)/3600'),
 'recency_hours':('hours','(Prediction time - latest eligible history time)/3600'),
 'no_observed_rows':('binary','1 if no eligible supplied-week rows; not lifetime novice status'),
 'no_buy':('binary','1 if eligible history has zero buy dollars'),
 'prior_conditional_profit_history_available':('binary','0 always: historical outcomes/accounting/time availability unverified')}
for col in history:
    raw=col[4:] if col.startswith('log_') else col
    units,definition=week_definitions[raw]
    if col.startswith('log_'):units='unitless';definition='ln(1 + '+definition+')'
    dictionary=pd.concat([dictionary,pd.DataFrame([{'column':'feature_week_history_'+col,'role':'feature',
      'units':units,'availability':'Observed supplied-week other-market fills strictly before prediction; coverage not globally verified',
      'definition':definition}])],ignore_index=True)
assert set(dictionary.column)==set(result.columns)
result.to_csv('results/full_week/wallet_market_full_week.csv',index=False)
dictionary.to_csv('results/full_week/wallet_market_full_week_feature_dictionary.csv',index=False)
with pd.option_context('display.max_colwidth',110,'display.max_rows',30):display(dictionary.tail(len(history.columns)))
display(result[['wallet','market','prediction_time','feature_week_history_fill_rows','feature_week_history_markets',
    'feature_week_history_buy_dollars','feature_week_history_buy_market_hhi','conditional_profit_label','confirmed_net_profit_label']].head(8))
provenance.update({'round':'03','duckdb':duckdb.__version__,'week_configuration':WEEK_CONFIG,
    'raw_week_files':file_info,'manifest':manifest,'manifest_sha256':sha256_stream(DATA/'weekly_trades/download_manifest.json'),
    'extra_input_hashes':{n:sha256_stream(DATA/n) for n in ['weekly_market_summary.csv','weekly_market_metadata_complete.csv','candidate_markets_preliminary.csv','candidate_markets_100plus_wallets.csv']},
    'pilot_multiset':multiset_report,'full_market_summary_all_fields_match':bool(compare.all_fields_match.all()),
    'decoded_rows':int(daily_audit.rows.sum()),'independent_history_pairs_checked':len(result),
    'legacy_pilot_history_observed_pairs':legacy_coverage,'week_history_observed_pairs':int(history.no_observed_rows.eq(0).sum()),
    'elapsed_seconds':time.perf_counter()-START,'week_stage_elapsed_seconds':time.perf_counter()-WEEK_START,
    'export_rows':len(result),'export_columns':list(result.columns),
    'export_sha256':sha256_stream(Path('results/full_week/wallet_market_full_week.csv'))})
Path('results/full_week/wallet_market_full_week_provenance.json').write_text(json.dumps(provenance,indent=2,default=str))
print('Saved integrated pilot table, dictionary and audit CSVs; elapsed seconds:',round(time.perf_counter()-START,2))


## 扩展至 2,661 个候选市场（条件性会计，非真实净利润）

这一步单独扩展到 239,434 条原始记录、96,184 个钱包–市场组合，保留重复记录，仍不验证结果映射或净利润。所有首次观察时点均有全周行为历史。历史计算利用按钱包–市场–时间聚合后的累计买入平方增量，再按钱包–时间排除整个当前时间批次；在目标市场首次观察时点之前该市场本来没有更早记录，因此严格全局历史自动排除了目标市场。此等价性只适用于首次观察预测点。

候选表的当前特征包含首次批次 C1–C6 与严格预测前市场环境；5 分钟相对价格只在原有试点表中计算，未在扩展表中声称已计算。未来库存、重复和条件利润列均为审计/标签，不能进入预测特征。>=100 钱包子集是覆盖描述，不是统计功效保证。更多市场并不消除标签和事件独立性问题。


In [ ]:
# Vectorized conditional accounting expansion; all input fills retained.
CANDIDATE_START=time.perf_counter()
# Release the independent pilot-check cache before full-universe history windows.
if 'wallet_week' in globals(): del wallet_week
if 'by_wallet' in globals(): del by_wallet
con.execute('DROP TABLE IF EXISTS eligible_history')
con.execute('DROP TABLE IF EXISTS pilot_wallet_week')
import gc
gc.collect()
candidate_meta=pd.read_csv(DATA/'candidate_markets_preliminary.csv')
con.register('expansion_keys',candidate_meta[['market']])
ct=con.execute('SELECT r.* FROM raw_week r JOIN expansion_keys k USING(market)').df()
ct['time']=pd.to_datetime(ct.time,utc=True)
assert len(ct)==239434
mapping=[]
for m in candidate_meta.itertuples(index=False):
    outcomes=json.loads(m.outcomes)
    assert len(outcomes)==2
    for k,o in enumerate(outcomes):mapping.append({'market':m.market,'outcome':o,'outcome_index':k,'payout':float(m.price_at_close if k==0 else 1-m.price_at_close)})
mp=pd.DataFrame(mapping)
ct=ct.merge(mp,on=['market','outcome'],how='left',validate='many_to_one')
assert ct.outcome_index.notna().all()
ct['signed_shares']=np.where(ct.side.eq('BUY'),ct['size'],-ct['size'])
ct['buy_dollars']=np.where(ct.side.eq('BUY'),ct.dollars,0.)
ct['sell_dollars']=np.where(ct.side.eq('SELL'),ct.dollars,0.)
ct['cashflow']=ct.sell_dollars-ct.buy_dollars
ct['base_pnl_contribution']=ct.cashflow+ct.signed_shares*ct.payout
ct['reverse_pnl_contribution']=ct.cashflow+ct.signed_shares*(1-ct.payout)
ct['duplicate_excess']=ct.duplicated(subset=keys).astype(int)
k=['wallet','market']
expanded=ct.groupby(k).agg(prediction_time=('time','min'),audit_only_final_fill_rows=('time','size'),
 label_final_observed_buy_dollars=('buy_dollars','sum'),label_final_observed_sell_dollars=('sell_dollars','sum'),
 label_final_observed_cashflow=('cashflow','sum'),label_assumed_zero_start_gross_pnl=('base_pnl_contribution','sum'),
 label_reversed_mapping_gross_pnl=('reverse_pnl_contribution','sum'),audit_only_duplicate_excess=('duplicate_excess','sum'))
q=ct.groupby(k+['outcome_index']).signed_shares.sum().unstack(fill_value=0).reindex(columns=[0,1],fill_value=0)
q.columns=['label_final_observed_net_shares_outcome0_zero_start','label_final_observed_net_shares_outcome1_zero_start']
expanded=expanded.join(q)
inv=ct.groupby(k+['outcome','time']).signed_shares.sum().reset_index().sort_values('time')
inv['balance']=inv.groupby(k+['outcome']).signed_shares.cumsum()
minimum=inv.groupby(k).balance.min().clip(upper=0).rename('audit_only_min_observed_inventory_zero_start')
expanded=expanded.join(minimum)
expanded['audit_only_negative_observed_inventory']=expanded.audit_only_min_observed_inventory_zero_start.lt(-CONFIG['inventory_epsilon']).astype(int)
dedup=ct[ct.duplicate_excess.eq(0)].groupby(k).base_pnl_contribution.sum().rename('label_dedup_assumption_gross_pnl')
expanded=expanded.join(dedup)
entry=ct.join(expanded[['prediction_time']],on=k)
entry=entry[entry.time.eq(entry.prediction_time)]
entry['buy_shares']=np.where(entry.side.eq('BUY'),entry['size'],0.)
entry['has_sell']=entry.side.eq('SELL').astype(int)
ef=entry.groupby(k).agg(feature_entry_buy_dollars=('buy_dollars','sum'),feature_entry_sell_dollars=('sell_dollars','sum'),
 feature_entry_buy_shares=('buy_shares','sum'),feature_entry_fill_rows=('time','size'),feature_entry_has_sell=('has_sell','max'))
ef['feature_entry_buy_vwap']=ef.feature_entry_buy_dollars/ef.feature_entry_buy_shares.replace(0,np.nan)
ef['feature_entry_no_buy']=ef.feature_entry_buy_shares.eq(0).astype(int)
ef['feature_entry_log_buy_dollars']=np.log1p(ef.feature_entry_buy_dollars)
ob=entry.groupby(k+['outcome_index']).buy_dollars.sum().unstack(fill_value=0).reindex(columns=[0,1],fill_value=0)
ef['feature_entry_outcome0_buy_dollar_share']=ob[0]/ef.feature_entry_buy_dollars.replace(0,np.nan)
ef['feature_entry_direction_concentration']=abs(2*ef.feature_entry_outcome0_buy_dollar_share-1)
ef['feature_entry_both_outcomes_bought']=(ob[0].gt(0)&ob[1].gt(0)).astype(int)
expanded=expanded.join(ef).reset_index()
assert len(expanded)==96184 and not expanded[k].duplicated().any()
for scenario,col in [('base','label_assumed_zero_start_gross_pnl'),('reverse','label_reversed_mapping_gross_pnl'),('dedup','label_dedup_assumption_gross_pnl')]:
    expanded['label_'+scenario+'_diagnostic_profit_sign']=expanded[col].gt(CONFIG['profit_epsilon']).astype(int)
    expanded['label_'+scenario+'_gross_state']=np.select([expanded[col]>CONFIG['profit_epsilon'],expanded[col]<-CONFIG['profit_epsilon']],['profit','loss'],default='breakeven')
expanded['conditional_profit_label']=expanded.label_base_diagnostic_profit_sign.astype('Int64').mask(expanded.audit_only_negative_observed_inventory.eq(1))
expanded['confirmed_net_profit_label']=pd.Series(pd.NA,index=expanded.index,dtype='Int64')
expanded['label_status']='unknown_unverified'
expanded['label_unknown_reasons']='settlement_mapping_unverified;opening_inventory_and_cost_basis_unverified;trade_coverage_unverified;fees_unavailable;transfers_split_merge_redemption_unavailable;historical_result_availability_unverified'
for reason in ['settlement_mapping_unverified','opening_inventory_and_cost_basis_unverified','trade_coverage_unverified','fees_unavailable','transfers_split_merge_redemption_unavailable','historical_result_availability_unverified']:
    expanded['label_reason_'+reason]=1
expanded['label_reason_negative_observed_inventory']=expanded.audit_only_negative_observed_inventory
expanded['label_reason_duplicate_looking_rows_without_unique_trade_id']=expanded.audit_only_duplicate_excess.gt(0).astype(int)
expanded.loc[expanded.audit_only_negative_observed_inventory.eq(1),'label_unknown_reasons']+=';negative_observed_inventory'
expanded.loc[expanded.audit_only_duplicate_excess.gt(0),'label_unknown_reasons']+=';duplicate_looking_rows_without_unique_trade_id'
assert expanded.loc[expanded.audit_only_negative_observed_inventory.eq(1),'label_unknown_reasons'].str.contains('negative_observed_inventory').all()
assert expanded.loc[expanded.audit_only_duplicate_excess.gt(0),'label_unknown_reasons'].str.contains('duplicate_looking_rows_without_unique_trade_id').all()
expanded['label_context_mapping_status']='unverified'
context=candidate_meta[['market','outcomes','price_at_close']].copy()
context['label_context_outcome0']=context.outcomes.map(lambda x:json.loads(x)[0])
context['label_context_outcome1']=context.outcomes.map(lambda x:json.loads(x)[1])
context['label_context_assumed_payout_outcome0']=context.price_at_close
context['label_context_assumed_payout_outcome1']=1-context.price_at_close
expanded=expanded.merge(context.drop(columns=['outcomes','price_at_close']),on='market',validate='many_to_one')
large=set(pd.read_csv(DATA/'candidate_markets_100plus_wallets.csv').market)
expanded['audit_only_in_100plus_wallet_market_subset']=expanded.market.isin(large).astype(int)
# New candidates use the same assumption boundary as pilots; no target is promoted to net profit.
assert expanded.confirmed_net_profit_label.isna().all()
assert np.isclose(expanded.label_assumed_zero_start_gross_pnl.sum(),ct.base_pnl_contribution.sum())
common=result.merge(expanded,on=k,suffixes=('_pilot','_expanded'),validate='one_to_one')
assert len(common)==2462
for col in ['label_assumed_zero_start_gross_pnl','label_reversed_mapping_gross_pnl','label_dedup_assumption_gross_pnl','label_final_observed_cashflow']:
    assert np.allclose(common[col+'_pilot'],common[col+'_expanded'],atol=1e-7,rtol=1e-10),col
print('Expanded conditional accounting:',len(ct),'fills,',len(expanded),'pairs across',expanded.market.nunique(),'markets')
# Strict-prior market context, excluding the complete entry timestamp batch.
mt=ct.groupby(['market','time']).agg(dollars=('dollars','sum')).sort_index().reset_index()
first_market_wallet=ct.groupby(['market','wallet']).time.min().reset_index()
new_wallets=first_market_wallet.groupby(['market','time']).size().rename('new_wallets').reset_index()
mt=mt.merge(new_wallets,on=['market','time'],how='left');mt['new_wallets']=mt.new_wallets.fillna(0)
mt['feature_strict_prior_market_dollars']=mt.groupby('market').dollars.cumsum()-mt.dollars
mt['feature_strict_prior_market_wallets']=mt.groupby('market').new_wallets.cumsum()-mt.new_wallets
mt['feature_observed_market_age_hours']=(mt.time-mt.groupby('market').time.transform('min')).dt.total_seconds()/3600
expanded=expanded.merge(mt[['market','time','feature_strict_prior_market_dollars','feature_strict_prior_market_wallets','feature_observed_market_age_hours']],left_on=['market','prediction_time'],right_on=['market','time'],validate='many_to_one').drop(columns='time')
expanded['feature_strict_prior_market_log_dollars']=np.log1p(expanded.feature_strict_prior_market_dollars)
expanded['feature_strict_prior_market_log_wallets']=np.log1p(expanded.feature_strict_prior_market_wallets)
expanded['feature_entry_utc_hour']=expanded.prediction_time.dt.hour
expanded['feature_entry_utc_weekday']=expanded.prediction_time.dt.dayofweek
expanded['feature_entry_utc_date']=expanded.prediction_time.dt.date.astype(str)
for k0 in (0,1):
    oq=entry[entry.outcome_index.eq(k0)].groupby(k).agg(buy_dollars=('buy_dollars','sum'),sell_dollars=('sell_dollars','sum'),buy_shares=('buy_shares','sum'),observed_net_shares_zero_start=('signed_shares','sum'))
    for field in oq:
        column='feature_entry_outcome'+str(k0)+'_'+field
        expanded=expanded.merge(oq[[field]].rename(columns={field:column}).reset_index(),on=k,how='left',validate='one_to_one')
        expanded[column]=expanded[column].fillna(0.)
    expanded['feature_entry_outcome'+str(k0)+'_buy_vwap']=expanded['feature_entry_outcome'+str(k0)+'_buy_dollars']/expanded['feature_entry_outcome'+str(k0)+'_buy_shares'].replace(0,np.nan)
expanded['feature_entry_total_dollars']=expanded.feature_entry_buy_dollars+expanded.feature_entry_sell_dollars
expanded['feature_entry_sell_dollar_fraction']=expanded.feature_entry_sell_dollars/expanded.feature_entry_total_dollars.replace(0,np.nan)
expanded['feature_entry_direction_hhi']=(1+expanded.feature_entry_direction_concentration**2)/2


In [ ]:
"""DuckDB strict-prior supplied-week history, without materializing raw rows in pandas."""


def build_weekly_entry_history(con, source_view='weekly_trades', output_table='week_history'):
    """Create history at every observed wallet/time. Identifiers must be simple names.

    Joining at a target market's first supplied-week timestamp implicitly excludes
    that target market: it has no earlier supplied-week rows. This does not hold
    for arbitrary later prediction times. Same-time rows across markets are all
    excluded. HHI includes positive cumulative BUY dollars; sell-only markets
    still count toward observed market breadth.
    """
    import re
    for name in (source_view, output_table):
        if not isinstance(name, str) or not re.fullmatch(r'[A-Za-z_][A-Za-z0-9_]*', name):
            raise ValueError('DuckDB identifiers must contain only letters, digits and underscores')
    # Materialize compact intermediate batches to avoid repeated raw scans.
    con.execute(f'''CREATE OR REPLACE TEMP TABLE _wh_market_batches AS
        SELECT wallet,market,time, count(*)::BIGINT AS n,
               sum(CASE WHEN side='BUY' THEN dollars ELSE 0 END)::DOUBLE AS b
        FROM {source_view} GROUP BY wallet,market,time''')
    con.execute('''CREATE OR REPLACE TEMP TABLE _wh_market_cumulative AS
        SELECT *, sum(b) OVER (PARTITION BY wallet,market ORDER BY time
                    ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS cum_b,
               row_number() OVER (PARTITION BY wallet,market ORDER BY time) AS market_batch
        FROM _wh_market_batches''')
    con.execute('''CREATE OR REPLACE TEMP TABLE _wh_wallet_batches AS
        SELECT wallet,time,sum(n)::BIGINT AS n,sum(b) AS b,
               sum(b*(2*cum_b-b)) AS square_delta,
               sum(CASE WHEN market_batch=1 THEN 1 ELSE 0 END)::BIGINT AS new_markets
        FROM _wh_market_cumulative GROUP BY wallet,time''')
    con.execute(f'''CREATE OR REPLACE TABLE {output_table} AS
        WITH prior AS (
            SELECT wallet,time,
                coalesce(sum(n) OVER w,0)::BIGINT AS rows_before,
                count(*) OVER w AS timestamps_before,
                coalesce(sum(new_markets) OVER w,0)::BIGINT AS markets_before,
                coalesce(sum(b) OVER w,0)::DOUBLE AS buys_before,
                coalesce(sum(square_delta) OVER w,0)::DOUBLE AS square_before,
                min(time) OVER w AS first_before, max(time) OVER w AS latest_before
            FROM _wh_wallet_batches
            WINDOW w AS (PARTITION BY wallet ORDER BY time
                ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING)
        ), metrics AS (
            SELECT *, CASE WHEN buys_before>0 THEN square_before/(buys_before*buys_before)
                           ELSE NULL END AS hhi,
                epoch(time-first_before)/3600.0 AS span_hours,
                epoch(time-latest_before)/3600.0 AS recency_hours
            FROM prior
        )
        SELECT wallet,time,
            rows_before AS feature_week_history_fill_rows,
            timestamps_before AS feature_week_history_distinct_timestamps,
            markets_before AS feature_week_history_markets,
            buys_before AS feature_week_history_buy_dollars,
            hhi AS feature_week_history_buy_market_hhi,
            first_before AS feature_week_history_first_time,
            latest_before AS feature_week_history_latest_time,
            span_hours AS feature_week_history_observed_span_hours,
            recency_hours AS feature_week_history_recency_hours,
            ln(1+recency_hours) AS feature_week_history_log_recency_hours,
            ln(1+rows_before) AS feature_week_history_log_fill_rows,
            ln(1+timestamps_before) AS feature_week_history_log_distinct_timestamps,
            ln(1+markets_before) AS feature_week_history_log_markets,
            ln(1+buys_before) AS feature_week_history_log_buy_dollars,
            (rows_before=0)::INTEGER AS feature_week_history_no_observed_rows,
            (buys_before=0)::INTEGER AS feature_week_history_no_buy,
            0::INTEGER AS feature_week_history_prior_conditional_profit_history_available
        FROM metrics''')
    for table in ('_wh_market_batches', '_wh_market_cumulative', '_wh_wallet_batches'):
        con.execute(f'DROP TABLE {table}')
    return output_table


def _test_fixture():
    import duckdb
    import math
    con=duckdb.connect()
    con.execute("""CREATE TABLE weekly_trades AS SELECT * FROM (VALUES
        ('w','a',TIMESTAMP '2026-10-01 00:00:00','BUY',2.0),
        ('w','b',TIMESTAMP '2026-10-01 00:00:00','BUY',3.0),
        ('w','a',TIMESTAMP '2026-10-01 01:00:00','BUY',2.0),
        ('w','c',TIMESTAMP '2026-10-01 01:00:00','SELL',9.0),
        ('w','d',TIMESTAMP '2026-10-01 02:00:00','BUY',0.0),
        ('s','a',TIMESTAMP '2026-10-01 00:00:00','SELL',1.0),
        ('s','b',TIMESTAMP '2026-10-01 01:00:00','BUY',0.0),
        ('s','c',TIMESTAMP '2026-10-01 02:00:00','SELL',1.0)
    ) AS t(wallet,market,time,side,dollars)""")
    build_weekly_entry_history(con)
    rows=con.execute('SELECT * FROM week_history ORDER BY wallet,time').fetchdf()
    w=rows[rows.wallet=='w'].reset_index(drop=True)
    assert w.loc[0,'feature_week_history_fill_rows']==0
    assert w.loc[1,'feature_week_history_fill_rows']==2
    assert w.loc[1,'feature_week_history_markets']==2
    assert w.loc[1,'feature_week_history_distinct_timestamps']==1
    assert math.isclose(w.loc[1,'feature_week_history_buy_market_hhi'],13/25)
    assert w.loc[2,'feature_week_history_fill_rows']==4
    assert w.loc[2,'feature_week_history_markets']==3
    assert math.isclose(w.loc[2,'feature_week_history_buy_market_hhi'],25/49)
    assert w.loc[2,'feature_week_history_recency_hours']==1
    s=rows[rows.wallet=='s']
    assert s.feature_week_history_buy_market_hhi.isna().all()
    assert s.feature_week_history_no_buy.eq(1).all()
    assert s.iloc[-1].feature_week_history_markets==2
    for bad in ('x; DROP TABLE x', 'schema.view', ''):
        try:
            build_weekly_entry_history(con, bad)
        except ValueError:
            pass
        else:
            raise AssertionError('unsafe identifier accepted')
    print('Weekly history fixture passed: ties, market breadth, repeated buys, sell-only, zero-dollar buys, identifier validation.')


if __name__ == '__main__':
    _test_fixture()


In [ ]:
# Global wallet-time windows: every candidate cutoff is its first supplied-week target-market timestamp.
build_weekly_entry_history(con,source_view='raw_week',output_table='week_history')
con.register('expanded_entry_keys',expanded[['wallet','market','prediction_time']])
expanded_history=con.execute('''SELECT e.wallet,e.market,h.* EXCLUDE(wallet,time)
 FROM expanded_entry_keys e JOIN week_history h ON e.wallet=h.wallet AND e.prediction_time=h.time''').df()
assert len(expanded_history)==len(expanded)
assert not expanded_history[['wallet','market']].duplicated().any()
for c in ['feature_week_history_first_time','feature_week_history_latest_time']:
    expanded_history[c]=pd.to_datetime(expanded_history[c],utc=True)
expanded=expanded.merge(expanded_history,on=['wallet','market'],validate='one_to_one')
# Independent direct-filter histories from original pilot path must match the new global-window algorithm.
verify=result[['wallet','market']+[c for c in result if c.startswith('feature_week_history_')]].merge(
    expanded[['wallet','market']+[c for c in expanded if c.startswith('feature_week_history_')]],
    on=['wallet','market'],suffixes=('_direct','_global'),validate='one_to_one')
assert len(verify)==2462
for c in [c for c in result if c.startswith('feature_week_history_')]:
    a=verify[c+'_direct'];b=verify[c+'_global']
    if 'first_time' in c or 'latest_time' in c:
        assert (a.eq(b)|(a.isna()&b.isna())).all(),c
    else:
        assert np.allclose(a.astype(float),b.astype(float),equal_nan=True,atol=1e-7,rtol=1e-9),c
print('Global-window candidate histories match all 2462 independent pilot cutoff histories.')
provenance['global_window_pilot_history_pairs_verified']=2462
con.execute('DROP TABLE week_history')


In [ ]:
# Release the database and intermediate frames before pandas string export.
# All SQL and the independent history comparison completed in the preceding cell.
candidate_raw_rows=len(ct) if 'ct' in globals() else candidate_raw_rows
if con is not None:
    con.close()
    con=None
import shutil, gc
shutil.rmtree(spill,ignore_errors=True)
for temporary_name in ['ct','entry','inv','ef','oq','ob','mt','first_market_wallet','new_wallets',
                       'common','expanded_history','verify','raw_pilot','multiset','a','b','h','w',
                       'wallet_week','by_wallet']:
    globals().pop(temporary_name,None)
gc.collect()
print('Released DuckDB and intermediate frames before candidate export.')
# Match global-window histories to candidates and reconcile every original pilot value.
assert len(expanded)==96184 and not expanded[['wallet','market']].duplicated().any()
expanded.to_parquet('results/full_week/wallet_market_candidates_full_week.parquet',index=False)
candidate_csv_path=Path('results/full_week/wallet_market_candidates_full_week.csv')
candidate_csv_temporary=candidate_csv_path.with_suffix('.csv.tmp')
expanded.to_csv(candidate_csv_temporary,index=False,chunksize=1000)
candidate_csv_temporary.replace(candidate_csv_path)
candidate_summary=expanded.groupby('market').agg(wallet_market_pairs=('wallet','size'),
 negative_inventory_pairs=('audit_only_negative_observed_inventory','sum'),
 duplicate_excess=('audit_only_duplicate_excess','sum'),conditional_labels=('conditional_profit_label','count'),
 conditional_profitable=('conditional_profit_label','sum'),assumed_gross_pnl=('label_assumed_zero_start_gross_pnl','sum'),
 reversed_mapping_gross_pnl=('label_reversed_mapping_gross_pnl','sum'))
candidate_summary.to_csv('results/full_week/candidate_market_accounting_summary.csv')
new_dictionary=[]
lookup=dictionary.set_index('column').to_dict('index')
for col in expanded:
    if col in lookup: row={'column':col,**lookup[col]}
    else:
        role='feature' if col.startswith('feature_') else ('audit only; never predictor' if col.startswith('audit_only_') else 'label/scenario/context; never predictor')
        row={'column':col,'role':role,'units':'binary' if 'sign' in col or 'subset' in col else 'categorical',
             'availability':'conditional observed accounting; not confirmed truth',
             'definition':col.replace('_',' ')+'; see inline vectorized accounting formulas'}
    new_dictionary.append(row)
pd.DataFrame(new_dictionary).to_csv('results/full_week/wallet_market_candidates_feature_dictionary.csv',index=False)
print('Expanded conditional cohort summary (all candidates vs >=100-wallet subset):')
for label,z in [('all 2661 candidates',expanded),('265 market subset',expanded[expanded.audit_only_in_100plus_wallet_market_subset.eq(1)])]:
    print(label,{'pairs':len(z),'markets':z.market.nunique(),'conditional_labels':z.conditional_profit_label.notna().sum(),
                 'confirmed_net_labels':z.confirmed_net_profit_label.notna().sum(),'negative_inventory_pairs':z.audit_only_negative_observed_inventory.sum()})
provenance['candidate_expansion']={'raw_rows':candidate_raw_rows,'pairs':len(expanded),'markets':expanded.market.nunique(),
    'conditional_labels':int(expanded.conditional_profit_label.notna().sum()),'confirmed_net_labels':0,
    'csv_sha256':sha256_stream(Path('results/full_week/wallet_market_candidates_full_week.csv')),
    'elapsed_seconds':time.perf_counter()-CANDIDATE_START}
Path('results/full_week/wallet_market_full_week_provenance.json').write_text(json.dumps(provenance,indent=2,default=str))
